# Gait Initiation Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PedroProspero23/gait-initiation-analysis/blob/main/gait_initiation_analysis.ipynb)

Biomechanical analysis of **gait initiation** from force-platform and motion-capture data (Qualisys/QTM text export).

For each trial, the notebook detects the **APA onset**, **foot off** and **foot contact**. It then computes:
- anticipatory postural adjustments (APA)
- swing time
- centre-of-mass velocities
- the **braking index**
- the mediolateral **margin of stability**
- a first-step spatial parameter

**To run:** `Runtime → Run all`. The sample file is downloaded automatically when it is not found in the working folder.


## 1. Settings


In [ ]:
import glob
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# Force-platform sampling frequency (Hz). Check it in QTM for each session.
FS_FORCE = 4420
# Motion-capture (camera) sampling frequency (Hz)
FS_CAMERA = 85
G = 9.81

# Columns per trial in the QTM export
COLS_PER_TRIAL = 27

# First stepping foot of each participant ('right' or 'left').
# Trials starting with the left foot are mirrored on the mediolateral axis.
SUBJECT_METADATA = {
    "HS_S1": {"first_foot": "right"},
    # "PD_S1": {"first_foot": "left"},
}

DATA_DIR = Path(".")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Sample data: downloaded from the repository if not present (e.g. when opened in Colab)
SAMPLE_FILES = ["HS_S1_GI_NS.txt"]
BASE_URL = "https://raw.githubusercontent.com/PedroProspero23/gait-initiation-analysis/main/"
for name in SAMPLE_FILES:
    if not (DATA_DIR / name).exists():
        print(f"Downloading sample file {name} ...")
        urllib.request.urlretrieve(BASE_URL + name, DATA_DIR / name)

# Files follow the convention <GROUP>_<SUBJECT>_GI_<CONDITION>[_<SESSION>].txt
data_files = sorted(glob.glob(str(DATA_DIR / "*_GI_*.txt")))
print(f"{len(data_files)} file(s) found:", [Path(f).name for f in data_files])


## 2. Data import

The QTM export has 5 header lines and the trials side by side: one frame column, then 27 columns per trial (FP1/FP2 forces, FP1/FP2 COP, heel and toe markers, centre of mass).


In [ ]:
TRIAL_COLUMNS = [
    "FP1_Fx", "FP1_Fy", "FP1_Fz",
    "FP2_Fx", "FP2_Fy", "FP2_Fz",
    "FP1_COP_X", "FP1_COP_Y", "FP1_COP_Z",
    "FP2_COP_X", "FP2_COP_Y", "FP2_COP_Z",
    "L_HEEL_X", "L_HEEL_Y", "L_HEEL_Z",
    "LTOES_X", "LTOES_Y", "LTOES_Z",
    "R_HEEL_X", "R_HEEL_Y", "R_HEEL_Z",
    "RTOES_X", "RTOES_Y", "RTOES_Z",
    "CoM_X", "CoM_Y", "CoM_Z",
]


def load_qtm_file(path):
    """Read a QTM .txt export.

    Returns the raw numeric columns (list of lists), a DataFrame with named
    columns (E1_FP1_Fz, E2_CoM_Y, ...) and the number of trials.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    # Split into columns (tab separated)
    columns = []
    for line in lines:
        values = line.split("\t")
        while len(columns) < len(values):
            columns.append([])
        for i, value in enumerate(values):
            columns[i].append(value)

    n_trials = int((len(columns) - 1) / COLS_PER_TRIAL)

    # Drop the header rows until numeric data are found
    while True:
        try:
            float(columns[1][0].strip())
            break
        except ValueError:
            for c in range(len(columns)):
                del columns[c][0]

    # Convert to float (empty cells become 0.0)
    numeric = [[float(v.strip()) if v.strip() != "" else 0.0 for v in col] for col in columns]

    names = ["Frame"]
    for t in range(1, n_trials + 1):
        names += [f"E{t}_{s}" for s in TRIAL_COLUMNS]
    while len(names) < len(numeric):
        names.append(f"Extra_{len(names)}")

    df = pd.DataFrame(numeric).T
    df.columns = names[: len(df.columns)]
    return numeric, df, n_trials


def mirror_mediolateral(df, n_trials):
    """Invert the mediolateral (Y) axis for participants starting with the left foot."""
    for t in range(1, n_trials + 1):
        for name in ["FP1_COP_Y", "FP2_COP_Y", "L_HEEL_Y", "LTOES_Y", "R_HEEL_Y", "RTOES_Y", "CoM_Y"]:
            col = f"E{t}_{name}"
            if col in df.columns:
                df[col] = -df[col]
    return df


## 3. Centre-of-mass kinematics from the ground reaction force

Body mass comes from the first second of quiet standing. The CoM acceleration is (F / m) − g, and the CoM velocity is obtained by numerical integration. Foot contact is the first non-zero vertical force on the second platform.


In [ ]:
def com_velocity(numeric, n_trials, fs=FS_FORCE):
    """Return vertical (z) and anteroposterior (x) CoM velocities for each trial."""
    vel_z, vel_x = [], []
    for i in range(n_trials):
        base = i * COLS_PER_TRIAL
        fz = [sum(v) for v in zip(numeric[base + 2 + 1], numeric[base + 5 + 1])]
        fx = [sum(v) for v in zip(numeric[base + 1], numeric[base + 3 + 1])]
        fz = [v for v in fz if v != 0.0]
        fx = [v for v in fx if v != 0.0]

        mass = float(np.mean(fz[0:fs]) / G)
        acc_z = [(f / mass) - G for f in fz]
        acc_x = [(f / mass) - (fx[0] / mass) for f in fx]

        vz, vx = [0], [0]
        for k in range(1, len(acc_z)):
            vz.append(acc_z[k] / fs + vz[k - 1])
            vx.append(acc_x[k] / fs + vx[k - 1])
        vel_z.append(vz)
        vel_x.append(vx)
    return vel_z, vel_x


def detect_foot_contact(numeric, n_trials):
    """First frame with non-zero vertical force on the second platform (FP2 Fz)."""
    contacts = []
    for i in range(n_trials):
        fz2 = numeric[i * COLS_PER_TRIAL + 5 + 1]
        for k, value in enumerate(fz2):
            if value != 0:
                contacts.append(k)
                break
    return contacts


## 4. Gait-initiation events and APA variables


In [ ]:
def detect_foot_off(df, n_trials, foot_contact, fs=FS_FORCE):
    """Foot off = start of the stable COP-Y plateau before foot contact.

    The COP-Y signal is scanned backwards from foot contact to the peak of its
    velocity; foot off is the first frame within 15 % of the rise amplitude
    from the final plateau (mean of the last 50 ms).
    """
    foot_off = []
    for i in range(n_trials):
        fc = foot_contact[i]
        cop_y = df[f"E{i + 1}_FP1_COP_Y"].values[: fc + 1]

        velocity = savgol_filter(cop_y, window_length=91, polyorder=2, deriv=1) * fs
        peak_idx = np.argmax(velocity)

        plateau = np.mean(cop_y[-int(0.05 * fs):])
        tolerance = 0.15 * (plateau - np.min(cop_y))

        fo = peak_idx
        for idx in range(len(cop_y) - 1, peak_idx, -1):
            if cop_y[idx] < (plateau - tolerance):
                fo = idx + 1
                break
        foot_off.append(fo)
    return foot_off


def detect_apa_onset_ml(df, n_trials, foot_contact, fs=FS_FORCE):
    """Mediolateral APA onset: last crossing of (local baseline mean - 2 SD)
    before the APA valley, scanning backwards."""
    onsets = []
    for i in range(n_trials):
        fc = foot_contact[i]
        cop_y = df[f"E{i + 1}_FP1_COP_Y"].values[: fc + 1]

        search_start = max(0, len(cop_y) - int(1.5 * fs))
        valley = search_start + np.argmin(cop_y[search_start:])

        base_start = max(0, valley - int(0.6 * fs))
        base_end = max(1, valley - int(0.3 * fs))
        threshold = np.mean(cop_y[base_start:base_end]) - 2.0 * np.std(cop_y[base_start:base_end])

        onset = base_end
        for idx in range(valley, 0, -1):
            if cop_y[idx] >= threshold:
                onset = idx
                break
        onsets.append(onset)
    return onsets


def detect_apa_onset_ap(df, n_trials, foot_contact, fs=FS_FORCE):
    """Anteroposterior APA onset: 5 % of the first COP-X ramp, scanning
    backwards from the APA valley to the preceding local maximum."""
    onsets = []
    for i in range(n_trials):
        fc = foot_contact[i]
        cop_x = df[f"E{i + 1}_FP1_COP_X"].values[: fc + 1]

        win_start = max(0, len(cop_x) - int(0.7 * fs))
        win_end = max(1, len(cop_x) - int(0.3 * fs))
        valley = win_start + np.argmin(cop_x[win_start:win_end])

        shoulder_start = max(0, valley - int(1.0 * fs))
        shoulder = shoulder_start + np.argmax(cop_x[shoulder_start:valley])

        threshold = cop_x[shoulder] - 0.05 * (cop_x[shoulder] - cop_x[valley])

        onset = shoulder
        for idx in range(valley, shoulder, -1):
            if cop_x[idx] >= threshold:
                onset = idx
                break
        onsets.append(onset)
    return onsets


def apa_variables(df, n_trials, onsets_ml, onsets_ap, foot_off, fs=FS_FORCE):
    """Duration (s) and amplitude (m) of the ML and AP APAs."""
    out = {"ml_dur": [], "ml_amp": [], "ap_dur": [], "ap_amp": []}
    for i in range(n_trials):
        cop_y = df[f"E{i + 1}_FP1_COP_Y"].values
        cop_x = df[f"E{i + 1}_FP1_COP_X"].values
        on_ml, on_ap, fo = onsets_ml[i], onsets_ap[i], foot_off[i]

        out["ml_dur"].append((fo - on_ml) / fs)
        out["ml_amp"].append(cop_y[on_ml] - np.min(cop_y[on_ml: fo + 1]))

        out["ap_dur"].append((fo - on_ap) / fs)
        end = min(len(cop_x), on_ap + int(0.8 * fs))
        out["ap_amp"].append(np.abs(cop_x[on_ap] - np.min(cop_x[on_ap:end])))
    return out


## 5. Braking index and margin of stability

- **Braking index**: (V<sub>z,min</sub> − V<sub>z,FC</sub>) / V<sub>z,min</sub>, the active braking of the CoM fall before foot contact.
- **Margin of stability**: the mediolateral base of support minus the extrapolated CoM, at foot contact. The XCoM is computed as y + v / ω, with ω = √(g / h<sub>CoM</sub>).


In [ ]:
def braking_index(vel_z, vel_x, foot_contact, window=1000):
    bi, peak_vx = [], []
    for i, fc in enumerate(foot_contact):
        vz_min = min(vel_z[i][0:fc])
        vz_fc = vel_z[i][fc]
        bi.append((vz_min - vz_fc) / vz_min)
        peak_vx.append(max(vel_x[i][fc:fc + window]))
    return bi, peak_vx


def margin_of_stability(df, n_trials, foot_contact, fs=FS_FORCE, fs_cam=FS_CAMERA):
    """Mediolateral margin of stability (m) at foot contact."""
    h_com = df["E1_CoM_Z"].iloc[0:fs_cam].mean()
    omega = np.sqrt(G / h_com)
    dt = 1 / fs_cam

    mos = []
    for i in range(n_trials):
        t = i + 1
        # Baseline: midpoint between the heels during the first second
        baseline = (df[f"E{t}_L_HEEL_Y"].iloc[0:fs_cam].mean() + df[f"E{t}_R_HEEL_Y"].iloc[0:fs_cam].mean()) / 2

        frame = int(round(foot_contact[i] * fs_cam / fs))
        y_com = df.loc[frame, f"E{t}_CoM_Y"] - baseline

        f = frame if frame > 0 else 1
        v_com = (df.loc[f + 1, f"E{t}_CoM_Y"] - df.loc[f - 1, f"E{t}_CoM_Y"]) / (2 * dt)
        xcom = y_com + v_com / omega

        bos = abs(df.loc[frame, f"E{t}_R_HEEL_Y"] - df.loc[frame, f"E{t}_L_HEEL_Y"])
        mos.append(bos - xcom)
    return mos


def step_parameter(df, n_trials, onsets_ml):
    """First-step spatial parameter: FP2 COP-X at FP1 unloading minus the minimum FP1 COP-X."""
    values = []
    for i in range(n_trials):
        t = i + 1
        t0 = int(onsets_ml[i])
        fp1_fz = df[f"E{t}_FP1_Fz"].values
        fp1_cop_x = df[f"E{t}_FP1_COP_X"].values
        fp2_cop_x = df[f"E{t}_FP2_COP_X"].values

        candidates = np.where(fp1_fz[t0:] == 0)[0]
        if len(candidates) == 0:
            values.append(np.nan)
            continue
        fo = candidates[0] + t0
        if np.isnan(fp2_cop_x[fo]):
            values.append(np.nan)
            continue
        values.append(fp2_cop_x[fo] - np.min(fp1_cop_x[t0:fo]))
    return values


## 6. Plots


In [ ]:
def plot_events(df, n_trials, foot_contact, foot_off, onsets_ml, onsets_ap, title, fs=FS_FORCE):
    colors = plt.cm.tab10(np.linspace(0, 1, n_trials))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for axis, ax, onsets, label in [("X", axes[0], onsets_ap, "anteroposterior"),
                                    ("Y", axes[1], onsets_ml, "mediolateral")]:
        for i in range(n_trials):
            fc = foot_contact[i]
            cop = df[f"E{i + 1}_FP1_COP_{axis}"].values[: fc + 1]
            time = (np.arange(len(cop)) - fc) / fs
            ax.plot(time, cop, color=colors[i], lw=1.5, label=f"Trial {i + 1}")
            ax.plot((onsets[i] - fc) / fs, cop[onsets[i]], "X", color=colors[i], ms=9, mec="black")
            ax.plot((foot_off[i] - fc) / fs, cop[foot_off[i]], "o", color=colors[i], ms=7, mec="black")
        ax.axvline(0, color="black", ls="--", lw=1.2)
        ax.set(xlabel="Time relative to foot contact (s)", ylabel=f"COP {axis} (m)",
               title=f"COP {axis} ({label}): X = APA onset, o = foot off")
        ax.grid(alpha=0.4)
    axes[1].legend(fontsize=8)
    fig.suptitle(title, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 7. Run the analysis


In [ ]:
all_results = []

for path in data_files:
    stem = Path(path).stem
    parts = stem.split("_")
    subject = f"{parts[0]}_{parts[1]}"
    condition = "MS" if "_MS_" in stem or stem.endswith("_MS") else "NS"
    first_foot = SUBJECT_METADATA.get(subject, {}).get("first_foot", "right")
    print(f"\n=== {stem} | subject {subject} | condition {condition} | first foot {first_foot} ===")

    numeric, df, n_trials = load_qtm_file(path)
    print(f"{n_trials} trials, {len(df)} frames")
    if first_foot == "left":
        df = mirror_mediolateral(df, n_trials)

    vel_z, vel_x = com_velocity(numeric, n_trials)
    foot_contact = detect_foot_contact(numeric, n_trials)
    foot_off = detect_foot_off(df, n_trials, foot_contact)
    onsets_ml = detect_apa_onset_ml(df, n_trials, foot_contact)
    onsets_ap = detect_apa_onset_ap(df, n_trials, foot_contact)
    apa = apa_variables(df, n_trials, onsets_ml, onsets_ap, foot_off)
    bi, peak_vx = braking_index(vel_z, vel_x, foot_contact)
    mos = margin_of_stability(df, n_trials, foot_contact)
    step = step_parameter(df, n_trials, onsets_ml)

    plot_events(df, n_trials, foot_contact, foot_off, onsets_ml, onsets_ap, stem)

    results = pd.DataFrame({
        "Subject": [subject] * n_trials,
        "Condition": [condition] * n_trials,
        "Trial": [f"Trial_{i + 1}" for i in range(n_trials)],
        "Braking_Index": bi,
        "Motor_Performance_Peak_Vel": peak_vx,
        "Margin_of_Stability_cm": [m * 100 for m in mos],
        "APA_AP_Duration_s": apa["ap_dur"],
        "APA_AP_Amplitude_m": apa["ap_amp"],
        "APA_ML_Duration_s": apa["ml_dur"],
        "APA_ML_Amplitude_m": apa["ml_amp"],
        "Swing_Time_s": [(fc - fo) / FS_FORCE for fc, fo in zip(foot_contact, foot_off)],
        "Vx_FO": [vel_x[i][foot_off[i]] for i in range(n_trials)],
        "Vy_FO": [vel_z[i][foot_off[i]] for i in range(n_trials)],
        "Step_Width_m": step,
    })
    results.to_excel(RESULTS_DIR / f"{stem}_results.xlsx", index=False)
    display(results.round(4))
    all_results.append(results)

summary = pd.concat(all_results, ignore_index=True)
summary.to_excel(RESULTS_DIR / "all_results.xlsx", index=False)
print(f"\nResults saved in {RESULTS_DIR.resolve()}")
